# 07. Arrays, hash tables and caches

**Goal.** Know how `list`, `dict` and `set` store data, so you can predict their speed and memory before you measure.

**You will be able to:**
1. Explain amortised O(1) append and why `insert(0, x)` is O(n).
2. Describe how CPython's `dict` stores entries, resolves collisions and resizes.
3. Build a dynamic array, an open-addressing map and an LRU cache and compare them with the built-ins.
4. Choose between `list`, `deque`, `array`, `dict` and `set` for a task.

## 1. Dynamic arrays

### How it works under the hood
- A `list` is a header plus one contiguous block of pointers to objects (`ob_item`).
- The header also keeps an `allocated` count, which is larger than `len`.
- `append` writes into spare capacity. When full, CPython allocates a larger block and copies the pointers. The growth rule is roughly `new = n + n/8 + 6`. Doubling gives the same amortised result: total copy work stays under a constant times n.
- `insert(0, x)` shifts every pointer one slot with `memmove`, so it costs O(n). `pop(0)` is the same.
- Indexing is pointer arithmetic: `ob_item[i]`, O(1). The block is contiguous, so scans are cache friendly, but the objects themselves live elsewhere.
- `collections.deque` is a doubly linked list of 64-slot blocks: O(1) at both ends, O(n) in the middle.
- `array.array` and NumPy arrays store raw machine values in one block, with no per-element object.

### Production use
- `list` for ordered, mixed, mutable data. `deque` for queues, sliding windows and BFS frontiers. `array` or NumPy for millions of numbers.
- Pre-size or build with comprehensions to avoid repeated resizing when n is known.

### Trade-offs

| Structure | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| `list` | 8 bytes per slot plus the objects | O(1) index and append, O(n) front insert | Default choice | Flexible, fast iteration | Front operations are slow, over-allocation |
| `deque` | Block overhead per 64 items | O(1) both ends, O(n) index | Queue-shaped code | Fast queues | Slow random access |
| `array('d')` | 8 bytes per value, no objects | O(1) index | Typed | Compact numeric storage | Single type, boxing on read |
| NumPy array | Raw values | Vectorised C loops | Different idiom | Fastest bulk maths | Not for mixed or growing data |

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "understanding_python").is_dir())
sys.path.insert(0, str(ROOT / "src"))
print("python", sys.version.split()[0], "| package root", ROOT.name)

python 3.13.6 | package root understanding-python


In [2]:
import sys
import time
from array import array
from collections import deque

from understanding_python.bench import best_time
from understanding_python.hashing import DynamicArray

grow, seen = [], []
for i in range(70):
    grow.append(i)
    capacity = (sys.getsizeof(grow) - sys.getsizeof([])) // 8
    if not seen or capacity != seen[-1][1]:
        seen.append((len(grow), capacity))
print("list: (length at resize, new capacity) ->", seen)

arr = DynamicArray()
for i in range(100):
    arr.append(i)
print("doubling array resizes to:", arr.resizes, "| 100 appends, final capacity", arr.capacity)

print(f"{'n':>8} {'append all':>12} {'insert(0) all':>14} {'deque.appendleft':>17}")
for n in (2_000, 4_000, 8_000, 16_000):
    def build_append():
        out = []
        for x in range(n):
            out.append(x)

    def build_front():
        out = []
        for x in range(n):
            out.insert(0, x)

    def build_deque():
        out = deque()
        for x in range(n):
            out.appendleft(x)

    print(f"{n:>8} {best_time(build_append) * 1e3:>10.2f} ms {best_time(build_front) * 1e3:>12.2f} ms {best_time(build_deque) * 1e3:>15.2f} ms")

n = 1_000_000
as_list = [float(i) for i in range(n)]
as_array = array("d", as_list)
print(f"1M floats: list {sys.getsizeof(as_list) / 1e6:.1f} MB of pointers + {n * 24 / 1e6:.0f} MB of float objects | array('d') {sys.getsizeof(as_array) / 1e6:.1f} MB")

queue = deque(range(100_000))
lst = list(range(100_000))
t_deque = best_time(lambda: [queue.popleft() for _ in range(5_000)], repeat=1)
t_list = best_time(lambda: [lst.pop(0) for _ in range(5_000)], repeat=1)
print(f"drain 5,000 items from the front of 100,000: deque {t_deque * 1e3:.2f} ms | list.pop(0) {t_list * 1e3:.2f} ms")

list: (length at resize, new capacity) -> [(1, 4), (5, 8), (9, 16), (17, 24), (25, 32), (33, 40), (41, 52), (53, 64), (65, 76)]
doubling array resizes to: [2, 4, 8, 16, 32, 64, 128] | 100 appends, final capacity 128
       n   append all  insert(0) all  deque.appendleft
    2000       0.09 ms         0.98 ms            0.10 ms
    4000       0.18 ms         3.49 ms            0.20 ms
    8000       0.37 ms        13.21 ms            0.40 ms


   16000       0.76 ms        54.74 ms            0.83 ms


1M floats: list 8.4 MB of pointers + 24 MB of float objects | array('d') 8.0 MB


drain 5,000 items from the front of 100,000: deque 0.33 ms | list.pop(0) 106.52 ms


## 2. Hash tables: `dict` and `set`

### How it works under the hood
- A hash table maps `hash(key)` to a slot. CPython's `dict` uses **open addressing**: on a collision it probes other slots in a pseudo-random sequence derived from the hash.
- Since 3.6 the `dict` is **compact**: a small sparse `indices` array points into a dense `entries` array of `(hash, key, value)`. This saves memory and preserves insertion order.
- The table resizes when two thirds full, so probe chains stay short and lookups average O(1).
- Keys must be hashable and equal keys must hash equal. Lists and dicts are unhashable because they can change after insertion.
- String hashes are randomised per process (`PYTHONHASHSEED`) to resist hash-flooding attacks. A bad `__hash__` that maps many keys to one value turns lookups into O(n).
- Deletion marks a slot as a **tombstone** so later probes continue past it. Tombstones count towards load until the next resize.

### Production use
- Indexing records by ID, counting events, deduplicating, caching, joins in ETL code, and every object's attribute table.
- `set` for fast membership and set algebra: allow-lists, tag matching, graph visited sets.

### Trade-offs

| Structure | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| `dict` / `set` | Higher per item than a list (index + entry) | O(1) average lookup, insert, delete | Direct | Fast keyed access, ordered dict | Worst case O(n) with bad hashes, memory hungry |
| Sorted list + `bisect` | Compact | O(log n) lookup, O(n) insert | Needs care | Ordered queries, range scans | Slow inserts |
| Separate chaining (`HashMap`) | Node or list per bucket | O(1) average | Easy to teach | Simple deletes | Pointer overhead, poor locality |
| Open addressing (`OpenAddressingMap`) | One flat array | O(1) average, degrades near full | Slightly intricate | Cache friendly | Tombstones, needs low load |

In [3]:
import random
import timeit

from understanding_python.hashing import OpenAddressingMap
from understanding_python.structures import HashMap

print("hash(1) =", hash(1), "| hash(1.0) =", hash(1.0), "| same key:", {1: "a"}.get(1.0))
print("tuple key ok:", {(51.5, -0.12): "London"}[(51.5, -0.12)])
try:
    {[1, 2]: "x"}
except TypeError as exc:
    print("unhashable:", exc)

rng = random.Random(7)
users = {f"user-{rng.randint(0, 5000):05d}": rng.randint(1, 99) for _ in range(20_000)}
mine = OpenAddressingMap()
chained = HashMap()
for k, v in users.items():
    mine.put(k, v)
    chained.put(k, v)
mine.probes = 0
assert all(mine.get(k) == v == chained.get(k) for k, v in users.items())
print(f"{len(users)} users | open addressing capacity {mine.capacity}, load {len(mine) / mine.capacity:.2f}, "
      f"{mine.probes / len(users):.2f} probes per lookup | chained buckets {chained.capacity}")

for k in list(users)[:1000]:
    mine.delete(k)
print("after 1,000 deletes:", len(mine), "live keys; deleted key reads as", mine.get(next(iter(users))))

caps = []
m = OpenAddressingMap(cap=8)
for i in range(60):
    before = m.capacity
    m.put(i, i)
    if m.capacity != before:
        caps.append((len(m), m.capacity))
print("resize points (live keys, new capacity):", caps)


class BadKey:
    def __init__(self, n):
        self.n = n

    def __hash__(self):
        return 1

    def __eq__(self, other):
        return self.n == other.n


class GoodKey(BadKey):
    def __hash__(self):
        return hash(self.n)


for count in (500, 1000, 2000):
    bad = {BadKey(i): i for i in range(count)}
    good = {GoodKey(i): i for i in range(count)}
    probe_bad, probe_good = BadKey(count - 1), GoodKey(count - 1)
    t_bad = timeit.timeit(lambda: bad[probe_bad], number=200) / 200
    t_good = timeit.timeit(lambda: good[probe_good], number=200) / 200
    print(f"{count:5} keys: constant hash {t_bad * 1e6:8.1f} us per lookup | good hash {t_good * 1e6:6.2f} us")

hash(1) = 1 | hash(1.0) = 1 | same key: a
tuple key ok: London
unhashable: unhashable type: 'list'
4909 users | open addressing capacity 8192, load 0.60, 1.71 probes per lookup | chained buckets 8192
after 1,000 deletes: 3909 live keys; deleted key reads as None
resize points (live keys, new capacity): [(6, 16), (11, 32), (22, 64), (43, 128)]
  500 keys: constant hash     43.1 us per lookup | good hash   0.31 us


 1000 keys: constant hash     86.4 us per lookup | good hash   0.30 us


 2000 keys: constant hash    180.3 us per lookup | good hash   0.30 us


## 3. LRU cache: a hash table plus a linked list

### How it works under the hood
- A cache needs O(1) lookup and O(1) eviction of the least recently used entry. A hash map alone finds entries but keeps no usage order.
- Pair the map with a **doubly linked list** ordered by recency. The map stores key to node. A hit unlinks the node and moves it to the front in O(1). An insert at capacity drops the tail node.
- A sentinel node removes the empty-list special cases.
- `functools.lru_cache` implements the same idea in C. `OrderedDict.move_to_end` gives a short pure-Python version.

### Production use
- In-process caches for config, parsed templates, DNS answers and ORM lookups.
- The same shape sits inside CPU caches, page caches and CDN edge nodes.

### Trade-offs

| Policy | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| LRU | Map plus 2 pointers per entry | O(1) get and put | Intuitive | Good on skewed traffic | A single scan can flush the cache |
| FIFO | Map plus queue | O(1) | Simplest | Cheap | Ignores popularity |
| TTL expiry | Map plus timestamps | O(1) | Matches freshness rules | Bounded staleness | Needs a clock and clean-up |
| No cache | None | Full cost each time | Simplest | No staleness | Repeated work |

In [4]:
import functools

from understanding_python.hashing import LRUCache

cache = LRUCache(3)
for key in ("a", "b", "c"):
    cache.put(key, key.upper())
cache.get("a")
cache.put("d", "D")
print("recency order (newest first):", cache.keys(), "| 'b' evicted:", cache.get("b") is None)

N_KEYS = 1000
rng = random.Random(11)
weights = [1 / (rank + 1) for rank in range(N_KEYS)]  # a few hot keys, a long tail
requests = rng.choices(range(N_KEYS), weights=weights, k=50_000)


def hit_rate(capacity):
    c = LRUCache(capacity)
    for key in requests:
        if c.get(key) is None:
            c.put(key, key)
    return c.hits / (c.hits + c.misses)


def stdlib_hit_rate(capacity):
    @functools.lru_cache(maxsize=capacity)
    def load(key):
        return key

    for key in requests:
        load(key)
    info = load.cache_info()
    return info.hits / (info.hits + info.misses)


print(f"{'capacity':>8} {'LRUCache':>9} {'lru_cache':>10}")
for cap in (10, 50, 100, 300, 1000):
    print(f"{cap:>8} {hit_rate(cap):>9.1%} {stdlib_hit_rate(cap):>10.1%}")

mine, stdlib = LRUCache(100), functools.lru_cache(maxsize=100)(lambda k: k)
t_mine = best_time(lambda: [mine.get(k) or mine.put(k, k) for k in requests[:20_000]], repeat=1)
t_std = best_time(lambda: [stdlib(k) for k in requests[:20_000]], repeat=1)
print(f"20,000 requests: pure Python LRUCache {t_mine * 1e3:.1f} ms | C lru_cache {t_std * 1e3:.1f} ms")

recency order (newest first): ['d', 'a', 'c'] | 'b' evicted: True
capacity  LRUCache  lru_cache
      10     21.2%      21.2%


      50     46.3%      46.3%
     100     57.8%      57.8%
     300     76.9%      76.9%
    1000     98.0%      98.0%


20,000 requests: pure Python LRUCache 15.1 ms | C lru_cache 2.9 ms
